In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Raw Data Exploration & Profiling
# MAGIC 
# MAGIC Profile what landed in the Bronze landing volume before we build Auto Loader pipelines.

# COMMAND ----------
from pyspark.sql import functions as F

CATALOG = "dbs_main_catalog"
LANDING = f"/Volumes/{CATALOG}/northwind_bronze/landing"

# Quick inventory: what's in our landing zone?
for source in ["orders", "products", "customers", "clickstream"]:
    files = dbutils.fs.ls(f"{LANDING}/{source}")
    total_bytes = sum(f.size for f in files)
    print(f"{source:12s} → {len(files):3d} files, {total_bytes / 1024 / 1024:.2f} MB")

In [0]:
# COMMAND ----------
# MAGIC %md ## Orders Profile

# COMMAND ----------
# Read all orders JSONL files
orders_raw = spark.read.json(f"{LANDING}/orders/")

print(f"Total orders: {orders_raw.count():,}")
orders_raw.printSchema()

# Sample 5 records
display(orders_raw.limit(5))

In [0]:
# COMMAND ----------
print("=== Data Quality Check on Raw Orders ===\n")

# 1. Duplicate order_ids — should be ~1%
total = orders_raw.count()
distinct_ids = orders_raw.select("order_id").distinct().count()
duplicates = total - distinct_ids
print(f"Duplicates: {duplicates:,} / {total:,} ({100 * duplicates / total:.2f}%)")

# 2. Null customer_ids — should be ~0.5%
null_customers = orders_raw.filter(F.col("customer_id").isNull()).count()
print(f"Null customer_ids: {null_customers:,} ({100 * null_customers / total:.2f}%)")

# 3. Negative totals — should be ~0.1%
negative_totals = orders_raw.filter(F.col("total_amount") < 0).count()
print(f"Negative totals: {negative_totals:,} ({100 * negative_totals / total:.4f}%)")

# 4. Schema drift: orders with discount_code (only day 5+)
if "discount_code" in orders_raw.columns:
    with_discount = orders_raw.filter(F.col("discount_code").isNotNull()).count()
    print(f"Orders with discount_code: {with_discount:,} (drift confirmed)")
else:
    print("⚠️  discount_code field NOT present — did you run day_offset=5+?")

# 5. Status distribution
print("\nStatus distribution:")
orders_raw.groupBy("status").count().orderBy(F.desc("count")).show()

In [0]:
# COMMAND ----------
# MAGIC %md ## Products Profile

# COMMAND ----------
# CSVs need explicit schema inference + header
products_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{LANDING}/products/"))

print(f"Total product rows (across all daily snapshots): {products_raw.count():,}")
products_raw.printSchema()

# How many unique products? Should be ~1000 since same products repeat daily
print(f"Unique product_ids: {products_raw.select('product_id').distinct().count():,}")

# Price changes per product across days — this is what drives our SCD2 logic later
price_changes = (products_raw
    .groupBy("product_id")
    .agg(F.countDistinct("price").alias("distinct_prices"))
    .filter(F.col("distinct_prices") > 1)
    .count())
print(f"Products with price changes across days: {price_changes:,}")

# Category distribution
products_raw.groupBy("category").count().orderBy(F.desc("count")).show()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, MapType

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("address", StructType([
        StructField("street", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("zip", StringType(), True),
        StructField("country", StringType(), True)
    ]), True),
    StructField("signup_date", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("preferred_language", StringType(), True),
    StructField("_snapshot_date", StringType(), True)
])

In [0]:
# COMMAND ----------
# MAGIC %md ## Customers Profile

# COMMAND ----------
customers_raw = (spark.read
                 .schema(customer_schema)
                 .parquet(f"{LANDING}/customers/"))

# Now these will work even if the folders are currently empty


display(customers_raw.limit(5))

print(f"Total customer rows (across all weekly snapshots): {customers_raw.count():,}")
customers_raw.printSchema()

# How many unique customers?
print(f"Unique customer_ids: {customers_raw.select('customer_id').distinct().count():,}")

# Snapshots present
print("\nSnapshots in landing:")
customers_raw.groupBy("_snapshot_date").count().orderBy("_snapshot_date").show()

# Detect address changes between snapshots — SCD2 driver
if customers_raw.select("_snapshot_date").distinct().count() > 1:
    addr_changes = (customers_raw
        .groupBy("customer_id")
        .agg(F.countDistinct(F.col("address.street")).alias("distinct_addresses"))
        .filter(F.col("distinct_addresses") > 1)
        .count())
    print(f"Customers with address changes across snapshots: {addr_changes:,}")
else:
    print("Only 1 snapshot — generate day_offset=7 to see SCD2 changes")

In [0]:
display(dbutils.fs.ls(f"{LANDING}/customers/"))

In [0]:
# COMMAND ----------
# MAGIC %md ## Clickstream Profile

# COMMAND ----------
# Read with rescued data column to capture malformed records
clickstream_raw = (spark.read
    .option("rescuedDataColumn", "_rescued_data")  # Captures parse failures
    .json(f"{LANDING}/clickstream/"))

total = clickstream_raw.count()
print(f"Total events: {total:,}")
clickstream_raw.printSchema()

# Anonymous user rate — should be ~30%
anonymous = clickstream_raw.filter(F.col("user_id").isNull()).count()
print(f"Anonymous events: {anonymous:,} ({100 * anonymous / total:.1f}%)")

# Out-of-order detection — events older than the day they "arrived"
# We don't have an arrival timestamp yet, so just check timestamp distribution
clickstream_raw.select(
    F.min("event_timestamp").alias("min_ts"),
    F.max("event_timestamp").alias("max_ts"),
).show(truncate=False)

# Event type distribution
clickstream_raw.groupBy("event_type").count().orderBy(F.desc("count")).show()

# Did we capture any malformed records in _rescued_data?
if "_rescued_data" in clickstream_raw.columns:
    rescued = clickstream_raw.filter(F.col("_rescued_data").isNotNull()).count()
    print(f"Rescued (malformed) records: {rescued:,}")

In [0]:
# What's in the customers folder?
print("Top-level customers folder:")
for f in dbutils.fs.ls(f"{LANDING}/customers"):
    print(f"  {f.name} | size={f.size} | isDir={f.isDir()}")

print("\nInside week_000/:")
for f in dbutils.fs.ls(f"{LANDING}/customers/week_000"):
    print(f"  {f.name} | size={f.size:,} bytes")

In [0]:
# Read WITHOUT explicit schema — let Parquet self-describe
customers_check = spark.read.parquet(f"{LANDING}/customers/")

print(f"Total customer rows: {customers_check.count():,}")
print(f"Unique customers: {customers_check.select('customer_id').distinct().count():,}")
print("\nSchema as written:")
customers_check.printSchema()

print("\nSample data:")
customers_check.select("customer_id", "first_name", "loyalty_tier", "_snapshot_date").show(5, truncate=False)

In [0]:
df = (spark.read
    .option("recursiveFileLookup", "true")
    .parquet(f"{LANDING}/customers/"))

print(f"Total customer rows: {df.count():,}")
df.printSchema()
df.select("customer_id", "first_name", "loyalty_tier", "_snapshot_date").show(5, truncate=False)